# 🚀 اجرای مدل Hermes 3 (8B) + رابط کاربری Open WebUI در Google Colab

این نوت‌بوک به شما اجازه می‌دهد تا بدون نیاز به پرداخت هزینه یا سرور اختصاصی، مدل قدرتمند **Hermes 3 (8B)** را به همراه پنل کاربری شبیه به ChatGPT (**Open WebUI**) بر روی پردازنده گرافیکی رایگان گوگل کولب (T4 GPU) اجرا کرده و از طریق لینک اختصاصی Cloudflare Tunnel به آن متصل شوید.

### گام ۱: بررسی وضعیت کارت گرافیک (GPU)
مطمئن شوید در منوی بالای کولب مسیر Runtime > Change runtime type را باز کرده و گزینه **T4 GPU** را فعال کرده‌اید.

In [ ]:
!nvidia-smi

### گام ۲: نصب پیش‌نیازها و نرم‌افزار Ollama

In [ ]:
# نصب پیش‌نیازها و آخرین نسخه Ollama
!apt-get update -qq && apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh

### گام ۳: اجرای سرویس Ollama در پس‌زمینه

In [ ]:
import subprocess
import time
import requests

print("در حال اجرای سرویس Ollama...")
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# بررسی سلامت و بالا آمدن سرویس
for _ in range(30):
    try:
        response = requests.get("http://127.0.0.1:11434/api/tags", timeout=2)
        if response.status_code == 200:
            print("✅ Ollama با موفقیت راه‌اندازی شد و آماده به کار است!")
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    raise RuntimeError("خطا: سرویس Ollama راه‌اندازی نشد.")

### گام ۴: دانلود مدل Hermes 3 (8B)

In [ ]:
!ollama pull hermes3:8b

### گام ۵: تست چت و پاسخ‌دهی به زبان فارسی

In [ ]:
import requests

resp = requests.post(
    "http://127.0.0.1:11434/api/chat",
    json={
        "model": "hermes3:8b",
        "messages": [
            {
                "role": "user",
                "content": "سلام! خودت را معرفی کن و فارسی جواب بده.",
            }
        ],
        "stream": False,
    },
    timeout=300,
)
resp.raise_for_status()
print(resp.json()["message"]["content"])


### گام ۶: نصب پنل گرافیکی Open WebUI

In [ ]:
!pip install -q uv
!uv venv --clear /content/webui_env --python 3.11
!uv pip install --python /content/webui_env/bin/python open-webui


### گام ۷: اجرای Open WebUI

In [ ]:
import os
import re
import shutil
import subprocess
import time
import urllib.request

print("🧹 ۱. پاکسازی پروسه‌های معلق قبلی...")
subprocess.run(
    "pkill -9 -f open-webui; pkill -9 -f cloudflared",
    shell=True,
    stderr=subprocess.DEVNULL,
)
time.sleep(2)


# ۲. بررسی و راه‌اندازی Ollama در صورت نیاز
def ensure_ollama():
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:11434/api/tags", timeout=2
        ) as res:
            if res.status == 200:
                print("✅ Ollama فعال است.")
                return
    except Exception:
        pass
    print("⚡ در حال راه‌اندازی Ollama...")
    subprocess.run("pkill -9 -f ollama", shell=True, stderr=subprocess.DEVNULL)
    subprocess.Popen(
        "OLLAMA_HOST=0.0.0.0:11434 nohup ollama serve > ollama.log 2>&1 &",
        shell=True,
    )
    time.sleep(4)
    print("✅ Ollama آماده شد.")


ensure_ollama()

# ۳. مطمئن شدن از نصب cloudflared
if not shutil.which("cloudflared"):
    print("⏳ در حال نصب ابزار Cloudflare...")
    subprocess.run(
        "wget -q -nc"
        " https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"
        " && dpkg -i cloudflared-linux-amd64.deb",
        shell=True,
        check=True,
    )

# ۴. اجرای Open WebUI با اتصال مستقیم به اولاما
webui_bin = "/content/webui_env/bin/open-webui"
print("🚀 در حال استارت Open WebUI روی پورت 8085...")
env = os.environ.copy()
env["WEBUI_AUTH"] = "True"
env["PORT"] = "8085"
env["OLLAMA_BASE_URL"] = "http://127.0.0.1:11434"

log_file = open("webui.log", "w")
subprocess.Popen(
    [webui_bin, "serve", "--host", "0.0.0.0", "--port", "8085"],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("⏳ ۲۰ ثانیه صبر برای آماده‌سازی فرانت‌اند...")
time.sleep(20)

# ۵. راه‌اندازی تانل استاندارد کلودفلر
print("🌐 در حال برقراری تانل و ساخت لینک...")
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8085"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

link_found = False
for line in cf_proc.stdout:
    if not link_found:
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            tunnel_url = m.group(0)
            print("\n" + "=" * 55)
            print("🎉 چت‌بات با موفقیت بالا اومد! کلیک کن:")
            print(f"👉 {tunnel_url}")
            print("=" * 55)
            print(
                "🟢 تانل فعال است (این سلول را متوقف نکنید و بگذارید در حال"
                " چرخش بماند)..."
            )
            link_found = True
